Mass changes the fuel rate a lot and none of our features says it. A heavier plane climbs slower with the same thrust, so the climb could give the mass back. First we check how many flights have a climb we can use, then try the mass on one flight and on all A359.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project="prc-data-challenge-2025")


def q(sql):
    return client.query(sql).to_dataframe()


traj = q("""
WITH sample AS (
  SELECT flight_id, aircraft_type, origin_icao, destination_icao, takeoff, landed
  FROM `prc-data-challenge-2025.prc2025.flightlist_train`
  ORDER BY FARM_FINGERPRINT(flight_id)
  LIMIT 6
)
SELECT
  s.flight_id,
  s.aircraft_type,
  s.origin_icao,
  s.destination_icao,
  TIMESTAMP_DIFF(s.landed, s.takeoff, SECOND) / 60 AS flight_min,
  TIMESTAMP_DIFF(TIMESTAMP_MICROS(DIV(f.timestamp, 1000)), s.takeoff, SECOND) / 60 AS min_since_takeoff,
  f.altitude,
  f.groundspeed,
  f.vertical_rate
FROM sample AS s
JOIN `prc-data-challenge-2025.prc2025.flights_train` AS f
  ON f.flight_id = s.flight_id
ORDER BY s.flight_id, min_since_takeoff
""")

In [ ]:
flights = traj.groupby("flight_id")
fig, axes = plt.subplots(nrows=flights.ngroups, figsize=(10, 3 * flights.ngroups))

for ax, (flight_id, g) in zip(axes, flights, strict=True):
    first = g.iloc[0]
    ax.plot(
        g["min_since_takeoff"],
        g["altitude"],
        marker=".",
        markersize=2,
        linestyle="none",
    )
    ax.axvline(0, color="grey", linestyle="--")
    ax.axvline(first["flight_min"], color="grey", linestyle="--")
    ax.set_title(
        f"{flight_id} {first['aircraft_type']} "
        f"{first['origin_icao']} -> {first['destination_icao']} ({first['flight_min']:.0f} min)"
    )
    ax.set_ylabel("altitude (ft)")

axes[-1].set_xlabel("minutes since takeoff")
fig.tight_layout()

6 flights picked with FARM_FINGERPRINT (same ones every run).  
Climb is there on 5 of 6, only MMGL -> KJFK starts at ~33,000 ft after 40 min.  
Not all clean though: LEMD -> SKBO has scattered points at the start and a ~7 min hole between 15,000 and 24,000 ft, KMIA -> SEQM small holes.  
Some climbs have level steps (LEMD -> SKBO ~15 min at 31,000 ft), the mass can't be read there.  
Top of climb between ~20 and ~45 min, so the climb is cut by altitude (25,000 ft) rather than a fixed time.  
Descent only on 2 flights, arrivals in Latin America are not covered. Isolated points in cruise look like the acars ones from the exploration.

In [ ]:
climb = q("""
WITH pts AS (
  SELECT
    flight_id,
    TIMESTAMP_MICROS(DIV(timestamp, 1000)) AS ts,
    altitude
  FROM `prc-data-challenge-2025.prc2025.flights_train`
  WHERE altitude BETWEEN -1000 AND 44000
),
per_flight AS (
  SELECT
    flight_id,
    COUNT(*) AS n_total,
    MIN(IF(altitude >= 25000, ts, NULL)) AS ts_25k
  FROM pts
  GROUP BY flight_id
),
climb AS (
  SELECT
    p.flight_id,
    p.ts,
    p.altitude,
    TIMESTAMP_DIFF(p.ts, LAG(p.ts) OVER (PARTITION BY p.flight_id ORDER BY p.ts), SECOND) AS gap_s
  FROM pts AS p
  JOIN per_flight AS f
    ON f.flight_id = p.flight_id
  WHERE p.ts <= f.ts_25k
),
climb_stats AS (
  SELECT
    flight_id,
    MIN(ts) AS first_ts,
    MIN_BY(altitude, ts) AS first_alt,
    MAX(gap_s) / 60 AS max_gap_min,
    COUNT(*) AS n_climb
  FROM climb
  GROUP BY flight_id
)
SELECT
  fl.flight_id,
  fl.aircraft_type,
  fl.origin_icao,
  f.n_total,
  c.n_climb,
  TIMESTAMP_DIFF(c.first_ts, fl.takeoff, SECOND) / 60 AS first_min,
  c.first_alt,
  c.max_gap_min,
  TIMESTAMP_DIFF(f.ts_25k, fl.takeoff, SECOND) / 60 AS min_to_25k
FROM `prc-data-challenge-2025.prc2025.flightlist_train` AS fl
LEFT JOIN per_flight AS f
  ON f.flight_id = fl.flight_id
LEFT JOIN climb_stats AS c
  ON c.flight_id = fl.flight_id
""")

In [ ]:
no_traj = climb["n_total"].isna()
no_25k = climb["n_total"].notna() & climb["min_to_25k"].isna()
has_climb = climb["min_to_25k"].notna()

pd.DataFrame(
    {
        "flights": [no_traj.sum(), no_25k.sum(), has_climb.sum()],
        "share": [no_traj.mean(), no_25k.mean(), has_climb.mean()],
    },
    index=["no trajectory", "never at 25,000 ft", "reaches 25,000 ft"],
)

Every flight has trajectory points and 99.8% are seen at 25,000 ft or more. Doesn't say yet if the climb itself is seen.

In [ ]:
cols = ["first_min", "first_alt", "max_gap_min", "min_to_25k", "n_climb"]
climb.loc[has_climb, cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

Half the flights have their first point under 2,000 ft, but a quarter start above ~31,500 ft (climb missed).  
max_gap_min only for 8,062 flights, the ~3,000 others have a single point before 25,000 ft so no gap.  
Median biggest gap 6 s, 95% under ~37 min.  
first_min and min_to_25k go negative (down to -194 min), points before takeoff? -> to check later if it matters.

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 4))

climb.loc[has_climb, "first_alt"].clip(upper=30000).hist(bins=60, ax=axes[0])
axes[0].set_xlabel("altitude of the first point (ft)")

climb.loc[has_climb, "max_gap_min"].clip(upper=30).hist(bins=60, ax=axes[1])
axes[1].set_xlabel("largest gap during the climb (min)")

for ax in axes:
    ax.set_yscale("log")
    ax.set_ylabel("flights")
fig.tight_layout()

y axis in log.  
First point altitude: big peak 0-3,000 ft (seen from takeoff), ~3,000 flights above 30,000 ft (climb missed), a bump around 8,000-13,000 ft.  
Biggest gap: almost everything under 1 min, very few between 5 and 25 min, ~500 flights at 30 min or more (last bar is clipped).  
Thresholds taken from that: first point under 3,000 ft, biggest gap under 5 min.

In [ ]:
usable = (climb["first_alt"] < 3000) & (climb["max_gap_min"] < 5)
usable.sum(), usable.mean()

5,762 flights (52%) have a usable climb. Mass is per flight so all their intervals get it, the others stay NaN.  
These are mostly departures from well covered areas -> check rank has about the same share.

In [ ]:
import numpy as np
from openap import Drag, Thrust, aero, prop

fid = "prc772222364"
pts = q(f"""
SELECT
  TIMESTAMP_MICROS(DIV(timestamp, 1000)) AS ts,
  altitude,
  groundspeed,
  vertical_rate,
  typecode
FROM `prc-data-challenge-2025.prc2025.flights_train`
WHERE flight_id = '{fid}'
ORDER BY ts
""")

In [ ]:
top = pts.loc[pts["altitude"] >= 25000, "ts"].min()

c = (
    pts[pts["ts"] <= top]
    .set_index("ts")[["altitude", "groundspeed", "vertical_rate"]]
    .resample("10s")
    .mean()
    .interpolate(limit=6)
)
c["dv_dt"] = c["groundspeed"].diff() * aero.kts / 10
c = c[c["altitude"].between(5000, 25000) & (c["vertical_rate"] > 500)].dropna()
len(c)

Climb cut at the first point above 25,000 ft, averaged every 10 s (smooths the noise and gives a regular step for the acceleration), small holes filled up to 1 min.  
Kept 5,000 to 25,000 ft only, below that flaps and gear are out and OpenAP's clean drag doesn't hold. Level steps removed with vertical_rate > 500 ft/min. 68 steps of 10 s left on this flight.

In [ ]:
ac = pts["typecode"].iloc[0].lower()
limits = prop.aircraft(ac)

alt = c["altitude"].values
gs = c["groundspeed"].values
vr = c["vertical_rate"].values
v = gs * aero.kts

thrust = Thrust(ac).climb(tas=gs, alt=alt, roc=vr)
climb_power_per_kg = aero.g0 * vr * aero.fpm + v * c["dv_dt"].values

masses = np.linspace(limits["oew"], limits["mtow"], 100)
errors = []
for m in masses:
    drag = Drag(ac).clean(mass=m, tas=gs, alt=alt, vs=vr)
    residual = (thrust - drag) * v - m * climb_power_per_kg
    errors.append(np.mean(residual**2))

best = masses[np.argmin(errors)]

fig, ax = plt.subplots()
ax.plot(masses / 1000, errors)
ax.axvline(best / 1000, color="red", linestyle="--")
ax.set_xlabel("mass tried (t)")
ax.set_ylabel("mean squared residual")
ax.set_title(
    f"{fid} {ac}: best mass {best / 1000:.0f} t (oew {limits['oew'] / 1000:.0f}, mtow {limits['mtow'] / 1000:.0f})"
)

Clear minimum, so one mass explains the climb better than the others. But 155 t is too light: only 27 t above oew (128 t) for a ~10 h flight.
From the exploration big twins burn 1.3-2.3 kg/s, so ~50-80 t of fuel over the flight, takeoff mass should be 200 t or more.
Probably because OpenAP gives max climb thrust and airlines climb with reduced thrust, less spare power looks like less mass. Groundspeed instead of airspeed adds some error too.
Flat between ~140 and ~170 t, so not very precise on this flight.
Might still be useful if the bias is the same for all flights -> check on many flights of one type that the mass goes up with route distance.

In [ ]:
def climb_points(pts):
    top = pts.loc[pts["altitude"] >= 25000, "ts"].min()
    c = (
        pts[pts["ts"] <= top]
        .set_index("ts")[["altitude", "groundspeed", "vertical_rate"]]
        .resample("10s")
        .mean()
        .interpolate(limit=6)
    )
    c["dv_dt"] = c["groundspeed"].diff() * aero.kts / 10
    return c[c["altitude"].between(5000, 25000) & (c["vertical_rate"] > 500)].dropna()


def estimate_mass(pts, ac, min_points=20):
    c = climb_points(pts)
    if len(c) < min_points:
        return np.nan

    limits = prop.aircraft(ac)
    alt = c["altitude"].values
    gs = c["groundspeed"].values
    vr = c["vertical_rate"].values
    v = gs * aero.kts

    thrust = Thrust(ac).climb(tas=gs, alt=alt, roc=vr)
    climb_power_per_kg = aero.g0 * vr * aero.fpm + v * c["dv_dt"].values

    drag = Drag(ac)
    masses = np.linspace(limits["oew"], limits["mtow"], 100)
    errors = [
        np.mean(
            (
                (thrust - drag.clean(mass=m, tas=gs, alt=alt, vs=vr)) * v
                - m * climb_power_per_kg
            )
            ** 2
        )
        for m in masses
    ]
    return masses[np.argmin(errors)]

In [ ]:
estimate_mass(pts, pts["typecode"].iloc[0].lower())

Cells above turned into climb_points and estimate_mass. Same 155 t on this flight. Returns NaN when less than 20 steps of 10 s are left (~3 min of climb).

In [ ]:
climb.loc[usable, "aircraft_type"].value_counts().head(15)

Usable climbs per type: A320 ~1,600, A20N ~1,600, A359 ~900, then B738 ~380 and the rest under 300.  
A359 picked: lots of usable climbs and a long-haul, so distances (and fuel loaded) vary a lot between flights. One type only, otherwise the mass would follow the distance just because big planes fly far (same trap as the OpenAP Spearman).

In [ ]:
config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter("ac", "STRING", "A359")]
)
climb_pts = client.query(
    """
WITH pts AS (
  SELECT
    f.flight_id,
    TIMESTAMP_MICROS(DIV(f.timestamp, 1000)) AS ts,
    f.altitude,
    f.groundspeed,
    f.vertical_rate
  FROM `prc-data-challenge-2025.prc2025.flights_train` AS f
  JOIN `prc-data-challenge-2025.prc2025.flightlist_train` AS fl
    ON fl.flight_id = f.flight_id
  WHERE fl.aircraft_type = @ac
    AND f.altitude BETWEEN -1000 AND 44000
),
top AS (
  SELECT flight_id, MIN(IF(altitude >= 25000, ts, NULL)) AS ts_25k
  FROM pts
  GROUP BY flight_id
)
SELECT p.*
FROM pts AS p
JOIN top AS t
  ON t.flight_id = p.flight_id
WHERE p.ts <= t.ts_25k
  AND p.altitude >= 4000
ORDER BY p.flight_id, p.ts
""",
    job_config=config,
).to_dataframe()

Only the climb points of the A359 (before 25,000 ft, above 4,000 ft), ~1.4 M rows. Type passed as a query parameter.

In [ ]:
usable_ids = climb.loc[usable, "flight_id"]

mass = climb_pts.groupby("flight_id").apply(lambda g: estimate_mass(g, "a359"))
mass = mass[mass.index.isin(usable_ids)]
mass.describe()

894 usable A359 flights, mass from 142 to 224 t, median ~176 t. Too light again (mtow 268 t), same bias as the B789.  
min is exactly oew (142.4 t), the bottom of the grid: for those flights the climb is slower than OpenAP allows even empty.

In [ ]:
from prc_fuel.features import route_distance
from prc_fuel.load import load

print("stuck at oew:", (mass == mass.min()).sum())

flights = load("train").groupby("flight_id").first()
test = pd.DataFrame({"mass": mass, "distance": route_distance(flights)}).dropna()

print("spearman:", test["mass"].corr(test["distance"], method="spearman"))
test.groupby(pd.qcut(test["distance"], 5))["mass"].median()

68 flights stuck at oew.  
Spearman mass vs route distance 0.18, median by distance fifth goes from 171.6 to 180.6 t, so ~9 t between ~6,000 and ~13,000 km where the fuel loaded should differ by ~35-40 t. Right direction, weak.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(test["distance"], test["mass"] / 1000, s=5, alpha=0.4)
ax.set_xlabel("route distance (km)")
ax.set_ylabel("estimated mass (t)")
ax.set_title("A359: estimated mass vs route distance")

Each vertical line is a route. Within one route the estimate spreads from ~140 to ~210 t, much more than the effect we look for. Row at 142 t = flights stuck at oew, on every route.

In [ ]:
from prc_fuel.features import duration_s

a359 = load("train").pipe(lambda d: d[d["flight_id"].isin(test.index)])
per_flight = (
    a359.assign(dur=duration_s(a359)).groupby("flight_id")[["fuel_kg", "dur"]].sum()
)

check = pd.DataFrame(
    {
        "mass": test["mass"],
        "rate": per_flight["fuel_kg"] / per_flight["dur"],
        "route": flights["origin_icao"] + "-" + flights["destination_icao"],
    }
).dropna()
check = check[check.groupby("route")["mass"].transform("size") >= 5]

for col in ["mass", "rate"]:
    check[col + "_in_route"] = check[col] - check.groupby("route")[col].transform(
        "median"
    )

check["mass_in_route"].corr(check["rate_in_route"], method="spearman")

Real test: within a route, do the flights estimated heavier burn more? Spearman between mass and fuel rate, both minus the route median, routes with 5+ flights: -0.008. Nothing.


The climb is there for about half the flights, but the mass we get from it is noise: too light, 70 t of spread on the same route, and no link with the fuel rate once the route is fixed. Not added as a feature.

Most likely causes: we only have groundspeed, so the wind during the climb is mixed in, and the thrust really used (reduced by the airlines) changes from flight to flight. Neither is in our data. Wind could come from public weather data (ERA5), noted in the roadmap as a maybe.